In [24]:
import pandas as pd

In [26]:
df = pd.read_csv("player_detections.csv")
df.iloc[100:150]

,frame,id,x1,y1,x2,y2
100,27,2,452.895447,666.327515,635.726074,835.816223
101,27,4,921.601318,53.515614,954.497070,131.876221
102,27,6,1492.503662,684.176331,1674.477783,876.202942
103,28,1,835.325989,144.023758,888.266541,244.477631
104,28,2,451.316223,666.012390,639.695923,835.154846
105,28,4,922.072998,54.046722,954.451599,131.939651
106,28,6,1486.211914,683.363159,1674.767700,872.979431
107,29,1,833.150635,143.523712,886.638000,244.912094
108,29,2,450.364899,665.719299,639.882935,834.817139
109,29,4,919.872437,54.445011,954.691528,132.011108


In [21]:
df.value_counts('id')

id
1     371
4     367
2     361
6     352
12     12
8       3
20      2
18      1
Name: count, dtype: int64

In [22]:
top_four_ids = df['id'].value_counts().head(4).index
df.loc[~df['id'].isin(top_four_ids), 'id'] = pd.NA
df = df.iloc[100:150]
df

,frame,id,x1,y1,x2,y2
100,27,2.0,452.895447,666.327515,635.726074,835.816223
101,27,4.0,921.601318,53.515614,954.497070,131.876221
102,27,6.0,1492.503662,684.176331,1674.477783,876.202942
103,28,1.0,835.325989,144.023758,888.266541,244.477631
104,28,2.0,451.316223,666.012390,639.695923,835.154846
105,28,4.0,922.072998,54.046722,954.451599,131.939651
106,28,6.0,1486.211914,683.363159,1674.767700,872.979431
107,29,1.0,833.150635,143.523712,886.638000,244.912094
108,29,2.0,450.364899,665.719299,639.882935,834.817139
109,29,4.0,919.872437,54.445011,954.691528,132.011108


In [23]:
# Fill missing ids by matching each NaN bbox to the closest bbox in the previous frame
def bbox_center(row):
    return ((row["x1"] + row["x2"]) / 2, (row["y1"] + row["y2"]) / 2)

nan_mask = df["id"].isna()

for idx, row in df.loc[nan_mask].iterrows():
    prev_frame = row["frame"] - 1
    prev_rows = df[(df["frame"] == prev_frame) & df["id"].notna()]
    
    if prev_rows.empty:
        continue
    
    cx, cy = bbox_center(row)
    prev_centers = prev_rows.apply(bbox_center, axis=1)
    distances = ((prev_centers.apply(lambda p: p[0]) - cx) ** 2 + (prev_centers.apply(lambda p: p[1]) - cy) ** 2) ** 0.5
    
    closest_idx = distances.idxmin()
    df.at[idx, "id"] = df.at[closest_idx, "id"]

df

,frame,id,x1,y1,x2,y2
100,27,2.0,452.895447,666.327515,635.726074,835.816223
101,27,4.0,921.601318,53.515614,954.497070,131.876221
102,27,6.0,1492.503662,684.176331,1674.477783,876.202942
103,28,1.0,835.325989,144.023758,888.266541,244.477631
104,28,2.0,451.316223,666.012390,639.695923,835.154846
105,28,4.0,922.072998,54.046722,954.451599,131.939651
106,28,6.0,1486.211914,683.363159,1674.767700,872.979431
107,29,1.0,833.150635,143.523712,886.638000,244.912094
108,29,2.0,450.364899,665.719299,639.882935,834.817139
109,29,4.0,919.872437,54.445011,954.691528,132.011108
